In [ ]:
import pandas as pd 
import baltic as bt
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import sys
from datetime import datetime
from matplotlib import cm
from matplotlib import colors as mcolors
from matplotlib.transforms import Affine2D
from matplotlib.patches import Rectangle
from matplotlib.patches import Wedge
from matplotlib.font_manager import FontProperties
from pathlib import Path

#Helper functions
def sentence_case(s):
    s = s.strip().lower()
    return s[:1].upper() + s[1:] if s else s


def norm_name(s):
    return ' '.join(str(s).strip().split())

#set helvetica neue font
plt.rcParams["font.family"] = "Helvetica Neue"

CANDIDATES = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(path for path in CANDIDATES if (path / "data" / "processed" / "sdm").exists())
SDM_DIR = REPO_ROOT / "data" / "processed" / "sdm"
hanta_hosts_tree = SDM_DIR / "hanta_host_tree_pruned.nwk"
taxonomy_path = SDM_DIR / "upham_taxonomy.csv"

min_alpha = 0.06
max_alpha = 0.40



In [ ]:
# -----------------------------
# 1) Load tree
# -----------------------------
host_ll = bt.loadNewick(hanta_hosts_tree, absoluteTime=False)
host_ll.sortBranches()
host_ll.setAbsoluteTime(2025)
host_ll.drawTree()

# -----------------------------
# 2) Load taxonomy + map tips -> family
# -----------------------------
upham_tax = pd.read_csv(taxonomy_path)
upham_tax["species"] = upham_tax["Species_Name"].str.replace("_", " ")
upham_tax["tip_clean"] = upham_tax["tiplabel"].str.replace("_", " ")

family_by_tip = dict(zip(upham_tax["tip_clean"], upham_tax["fam"]))
family_by_species = dict(zip(upham_tax["species"], upham_tax["fam"]))

leaves = [b for b in host_ll.Objects if getattr(b, "branchType", None) == "leaf"]
leaf_species = [b.name.replace("_", " ") for b in leaves]
leaf_fams = [family_by_tip.get(s) or family_by_species.get(s) or "Unknown" for s in leaf_species]

unique_fams = sorted(set(leaf_fams))
print("Families in plot:", len(unique_fams), "(Unknown:", leaf_fams.count("Unknown"), ")")

palette = cm.get_cmap("tab20", len(unique_fams))
family_colors = {fam: palette(i) for i, fam in enumerate(unique_fams)}

# -----------------------------
# 3) Plot circular tree
# -----------------------------
# Opening control: keep covariate label lane vertically centered at top
gap_frac = 20.0 / 360.0
gap_center_deg = 90.0

circ_frac = 1.0 - gap_frac
gap_center_frac = gap_center_deg / 360.0
circ_start = (gap_center_frac - (1.0 - gap_frac / 2.0)) % 1.0

inward = 0.05 * host_ll.treeHeight

fig = plt.figure(figsize=(15, 15))
ax_host = fig.add_axes([0.02, 0.02, 0.96, 0.96])

host_ll.plotCircularTree(
    ax=ax_host,
    circStart=circ_start,
    circFrac=circ_frac,
    colour=lambda k: "k",
    inwardSpace=inward,
    width=1
)

host_ll.plotCircularPoints(
    ax=ax_host,
    circStart=circ_start,
    circFrac=circ_frac,
    size=25,
    colour=lambda k: family_colors.get(
        family_by_tip.get(k.name.replace("_", " "))
        or family_by_species.get(k.name.replace("_", " "))
        or "Unknown",
        (0.2, 0.2, 0.2, 1.0),
    ),
    inwardSpace=inward,
    outline=False,
    alpha=max_alpha
)

# Keep the working viewport so the tree is not tiny
ax_host.set_aspect("equal", adjustable="box")
ax_host.set_xlim(-1.1, 1.1)
ax_host.set_ylim(-1.1, 1.1)

# -----------------------------
# 4) Build contiguous family runs using baltic transform + matplotlib angle conversion
# -----------------------------
# baltic circular uses:
#   X = sin(theta_b), Y = cos(theta_b)  (clockwise-positive from top)
# matplotlib Wedge uses standard polar:
#   X = cos(theta_m), Y = sin(theta_m)  (CCW-positive from right)
# conversion: theta_m = pi/2 - theta_b
leaf_rows = []
for b in leaves:
    sp = b.name.replace("_", " ")
    fam = family_by_tip.get(sp) or family_by_species.get(sp) or "Unknown"
    theta_b = (2 * np.pi * circ_start) + (2 * np.pi * circ_frac * (b.y / host_ll.ySpan))
    theta_m = (np.pi / 2.0 - theta_b) % (2 * np.pi)
    leaf_rows.append((theta_m, sp, fam, b))

# Order leaves in matplotlib angular space (matches Wedge direction)
leaf_rows.sort(key=lambda t: t[0])

thetas = np.array([r[0] for r in leaf_rows])
leaf_species_ord = [r[1] for r in leaf_rows]
leaf_fams_ord = [r[2] for r in leaf_rows]
leaves_ord = [r[3] for r in leaf_rows]

n = len(leaves_ord)
if n > 1:
    theta_wrap = np.r_[thetas, thetas[0] + 2 * np.pi]
    dtheta = np.median(np.diff(theta_wrap))
else:
    dtheta = 0.0

runs = []
if n > 0:
    s_idx = 0
    for i in range(1, n):
        if leaf_fams_ord[i] != leaf_fams_ord[i - 1]:
            runs.append((s_idx, i - 1, leaf_fams_ord[i - 1]))
            s_idx = i
    runs.append((s_idx, n - 1, leaf_fams_ord[-1]))

    # merge first/last only when tree is effectively a full circle
    if circ_frac >= 0.999 and len(runs) > 1 and runs[0][2] == runs[-1][2]:
        first = runs[0]
        last = runs[-1]
        runs = [(last[0], first[1], first[2])] + runs[1:-1]

# -----------------------------
# 5) Draw annular tiles (grey/white alternating, radial fade)
#    in normalized axis-space so it matches your fixed limits
# -----------------------------
r_outer = 1.01  # tiny overshoot so no visible gap
r_inner = 0.12
n_slices = 40
min_alpha = 0.06
max_alpha = 0.40

unique_fams = sorted(set(leaf_fams_ord))
palette = cm.get_cmap("tab20", len(unique_fams))
family_colors = {fam: palette(i) for i, fam in enumerate(unique_fams)}

for ridx, (i0, i1, fam) in enumerate(runs):
    tile_color = family_colors.get(fam, (0.7, 0.7, 0.7, 1.0))

    t1 = thetas[i0] - dtheta / 2
    t2 = thetas[i1] + dtheta / 2
    if i0 > i1:  # wrapped run
        t2 += 2 * np.pi

    for j in range(n_slices):
        frac = j / (n_slices - 1)
        rr0 = r_outer - (j + 1) * (r_outer - r_inner) / n_slices
        rr1 = r_outer - j * (r_outer - r_inner) / n_slices
        alpha = max_alpha - (max_alpha - min_alpha) * frac

        w = Wedge(
            center=(0, 0),
            r=rr1,
            theta1=np.degrees(t1),
            theta2=np.degrees(t2),
            width=(rr1 - rr0),
            facecolor=tile_color,
            edgecolor="none",
            linewidth=0,
            alpha=alpha,
            zorder=1
        )
        ax_host.add_patch(w)

# Put tree/points above tiles
for coll in ax_host.collections:
    coll.set_clip_on(False)
    coll.set_zorder(3)

# -----------------------------
# 6) Final styling
# -----------------------------
for spine in ax_host.spines.values():
    spine.set_visible(False)
ax_host.set_xticks([])
ax_host.set_yticks([])

# Tip coords for downstream links (if needed later)
host_tip = {b.name.replace("_", " "): (b.x, b.y) for b in leaves}

label_r = r_outer * .5  # move labels just outside tiles
flipped_families = {"didelphidae", "tenrecidae", "talpidae", "soricidae"}

for i0, i1, fam in runs:
    # skip tiny sectors
    count = (i1 - i0 + 1) if i0 <= i1 else (len(thetas) - i0 + i1 + 1)
    if count < 3:
        continue

    # sector midpoint angle
    t1 = thetas[i0]
    t2 = thetas[i1] if i1 >= i0 else (thetas[i1] + 2 * np.pi)
    tm = (t1 + t2) / 2.0

    x = label_r * np.cos(tm)
    y = label_r * np.sin(tm)

    rot = np.degrees(tm)
    if 90 < rot < 270:
        rot += 180
        ha = "right"
    else:
        ha = "left"

    if str(fam).strip().lower() in flipped_families:
        rot += 180
        ha = "left" if ha == "right" else "right"

    ax_host.text(
        x, y, sentence_case(fam),
        rotation=rot, rotation_mode="anchor",
        ha=ha, va="center", fontsize=16, color="black", zorder=5
    )

# -----------------------------
# 7) Real outer tracks (PDP Spearman heatmap + GBIF counts)
# -----------------------------
from matplotlib import cm, colors as mcolors
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.spatial.distance import pdist


def normalize_species_name(name):
    return " ".join(str(name).replace("_", " ").strip().lower().split())


# Input files
pdp_wide_path = SDM_DIR / "Complete_PDP" / "pdp_spearman_all_species_wide.csv"
gbif_points_path = SDM_DIR / "gbif_points_per_species.csv"
arha_synonym_path = SDM_DIR / "pdp_unmatched_arha_synonym_candidates.csv"

# Read + validate
pdp_wide = pd.read_csv(pdp_wide_path)
gbif_points = pd.read_csv(gbif_points_path)

required_pdp_cols = {"species"}
required_gbif_cols = {"species", "n_records"}
if not required_pdp_cols.issubset(pdp_wide.columns):
    missing = sorted(required_pdp_cols - set(pdp_wide.columns))
    raise ValueError(f"Missing required columns in {pdp_wide_path}: {missing}")
if not required_gbif_cols.issubset(gbif_points.columns):
    missing = sorted(required_gbif_cols - set(gbif_points.columns))
    raise ValueError(f"Missing required columns in {gbif_points_path}: {missing}")

if "arha_unique_points" not in gbif_points.columns:
    print("Warning: arha_unique_points not found in GBIF table; using zeros.")
    gbif_points["arha_unique_points"] = 0

# Use all covariates from wide table (23 expected)
human_covariates = [
    "EC2019_working",
    "F182013.v4c_web.avg_vis",
    "gpw_v4_population_density_rev11_2020_30s",
    "wildareas-v3-2009-human-footprint_geo",
]
temperature_covariates = [
    "wc2.1_30s_bio_1",
    "wc2.1_30s_bio_2",
    "wc2.1_30s_bio_3",
    "wc2.1_30s_bio_4",
    "wc2.1_30s_bio_5",
    "wc2.1_30s_bio_6",
    "wc2.1_30s_bio_7",
    "wc2.1_30s_bio_8",
    "wc2.1_30s_bio_9",
    "wc2.1_30s_bio_10",
    "wc2.1_30s_bio_11",
]
precipitation_covariates = [
    "wc2.1_30s_bio_12",
    "wc2.1_30s_bio_13",
    "wc2.1_30s_bio_14",
    "wc2.1_30s_bio_15",
    "wc2.1_30s_bio_16",
    "wc2.1_30s_bio_17",
    "wc2.1_30s_bio_18",
    "wc2.1_30s_bio_19",
]
all_covariates = [c for c in pdp_wide.columns if c != "species"]
manual_covariate_order = human_covariates + temperature_covariates + precipitation_covariates
missing_covs = [c for c in manual_covariate_order if c not in all_covariates]
if missing_covs:
    raise ValueError(f"Missing expected covariates in {pdp_wide_path}: {missing_covs}")

cluster_covariates = False
covariate_order = manual_covariate_order.copy()
print(f"Covariates included in heatmap: {len(covariate_order)}")

# Tree species order from current circular plot
tips_for_tracks = leaves_ord
species_order = [b.name.replace("_", " ") for b in tips_for_tracks]
species_order_norm = [normalize_species_name(s) for s in species_order]
n_species = len(species_order)

# Family lookup in normalized space (for bar colors)
species_to_family_norm = {
    normalize_species_name(b.name.replace("_", " ")): (
        family_by_tip.get(b.name.replace("_", " "))
        or family_by_species.get(b.name.replace("_", " "))
        or "Unknown"
    )
    for b in tips_for_tracks
}

# Build ArHa synonym mapping for taxonomy harmonization
# expected columns: species_pdp_unmatched, arha_canonical_species
arha_synonym_map = {}
try:
    arha_syn = pd.read_csv(arha_synonym_path)
    req_syn_cols = {"species_pdp_unmatched", "arha_canonical_species"}
    if req_syn_cols.issubset(arha_syn.columns):
        arha_synonym_map = dict(zip(
            arha_syn["species_pdp_unmatched"].map(normalize_species_name),
            arha_syn["arha_canonical_species"].map(normalize_species_name),
        ))
    else:
        missing = sorted(req_syn_cols - set(arha_syn.columns))
        print(f"Warning: {arha_synonym_path} missing columns {missing}; no synonym remapping applied.")
except FileNotFoundError:
    print(f"Warning: {arha_synonym_path} not found; no synonym remapping applied.")


def apply_arha_synonym(name_norm):
    return arha_synonym_map.get(name_norm, name_norm)

# Normalize + align PDP data
pdp_wide = pdp_wide.copy()
pdp_wide["species_norm_raw"] = pdp_wide["species"].map(normalize_species_name)
pdp_wide["species_norm"] = pdp_wide["species_norm_raw"].map(apply_arha_synonym)

n_pdp_remapped = int((pdp_wide["species_norm"] != pdp_wide["species_norm_raw"]).sum())
if n_pdp_remapped > 0:
    print(f"ArHa synonym remaps applied to PDP species: {n_pdp_remapped}")

pdp_wide = pdp_wide.drop_duplicates(subset=["species_norm"], keep="first")

# apply same canonicalization on tree labels for joining
species_order_norm_mapped = [apply_arha_synonym(s) for s in species_order_norm]

pdp_matrix = (
    pdp_wide.set_index("species_norm")[covariate_order]
    .reindex(species_order_norm_mapped)
    .fillna(0.0)
)

if cluster_covariates and len(covariate_order) > 1:
    clustered_input = pdp_matrix[covariate_order].T.to_numpy()
    clustered_dist = pdist(clustered_input, metric="correlation")
    if np.isnan(clustered_dist).any():
        clustered_dist = np.nan_to_num(clustered_dist, nan=1.0, posinf=1.0, neginf=1.0)
    clustered_linkage = linkage(clustered_dist, method="average")
    clustered_idx = leaves_list(clustered_linkage)
    covariate_order = [covariate_order[i] for i in clustered_idx]
    pdp_matrix = pdp_matrix[covariate_order]
    print("Covariates reordered by hierarchical clustering.")
    print("Clustered order:", covariate_order)
else:
    print("Using manual covariate order:", covariate_order)

# Normalize + align GBIF and ArHa point counts
gbif_points = gbif_points.copy()
gbif_points["species_norm_raw"] = gbif_points["species"].map(normalize_species_name)
gbif_points["species_norm"] = gbif_points["species_norm_raw"].map(apply_arha_synonym)

gbif_points["n_records"] = pd.to_numeric(gbif_points["n_records"], errors="coerce").fillna(0.0)
gbif_points["arha_unique_points"] = pd.to_numeric(gbif_points["arha_unique_points"], errors="coerce").fillna(0.0)

n_gbif_remapped = int((gbif_points["species_norm"] != gbif_points["species_norm_raw"]).sum())
if n_gbif_remapped > 0:
    print(f"ArHa synonym remaps applied to GBIF species: {n_gbif_remapped}")

gbif_points = gbif_points.drop_duplicates(subset=["species_norm"], keep="first")

gbif_counts = (
    gbif_points.set_index("species_norm")["n_records"]
    .reindex(species_order_norm_mapped)
    .fillna(0.0)
)

arha_counts = (
    gbif_points.set_index("species_norm")["arha_unique_points"]
    .reindex(species_order_norm_mapped)
    .fillna(0.0)
)

# Validation checks
assert pdp_matrix.shape[0] == n_species, "PDP matrix row count != tree tip count"
assert gbif_counts.shape[0] == n_species, "GBIF count vector length != tree tip count"
assert arha_counts.shape[0] == n_species, "ArHa count vector length != tree tip count"
assert not np.isnan(pdp_matrix.values).any(), "NaN values remain in PDP matrix"

# Missing-host reports
tree_species_df = pd.DataFrame({
    "species_tree": species_order,
    "species_tree_norm": species_order_norm_mapped,
})

pdp_species_set = set(pdp_wide["species_norm"].dropna())
gbif_species_set = set(gbif_points["species_norm"].dropna())

missing_summary_df = tree_species_df.copy()
missing_summary_df["in_pdp"] = missing_summary_df["species_tree_norm"].map(lambda x: x in pdp_species_set)
missing_summary_df["in_gbif"] = missing_summary_df["species_tree_norm"].map(lambda x: x in gbif_species_set)
missing_summary_df["pdp_fill_value"] = np.where(missing_summary_df["in_pdp"], np.nan, 0)
missing_summary_df["gbif_fill_value"] = np.where(missing_summary_df["in_gbif"], np.nan, 0)
missing_pdp_mask = (~missing_summary_df["in_pdp"]).to_numpy()

missing_pdp_df = missing_summary_df.loc[~missing_summary_df["in_pdp"], [
    "species_tree", "in_pdp", "in_gbif", "pdp_fill_value", "gbif_fill_value"
]].copy()
missing_gbif_df = missing_summary_df.loc[~missing_summary_df["in_gbif"], [
    "species_tree", "in_pdp", "in_gbif", "pdp_fill_value", "gbif_fill_value"
]].copy()
missing_summary_df = missing_summary_df[[
    "species_tree", "in_pdp", "in_gbif", "pdp_fill_value", "gbif_fill_value"
]].copy()

SDM_DIR.mkdir(parents=True, exist_ok=True)
missing_pdp_df.to_csv(SDM_DIR / "missing_hosts_pdp.csv", index=False)
missing_gbif_df.to_csv(SDM_DIR / "missing_hosts_gbif.csv", index=False)
missing_summary_df.to_csv(SDM_DIR / "missing_hosts_summary.csv", index=False)

print(f"Missing hosts in PDP: {len(missing_pdp_df)}")
print(f"Missing hosts in GBIF: {len(missing_gbif_df)}")

# Angular sector edges from transformed tip angles (same basis as tile wedges)
if n_species > 1:
    theta_wrap_tracks = np.r_[thetas, thetas[0] + 2 * np.pi]
    dtheta_tracks = np.median(np.diff(theta_wrap_tracks))
else:
    dtheta_tracks = 2 * np.pi * circ_frac

theta_edges = np.r_[thetas - dtheta_tracks / 2.0, thetas[-1] + dtheta_tracks / 2.0]

# Gap settings derived from transformed tip angles (same basis as wedges)
# This ensures heatmap/bar gap exactly matches the tree opening.
if n > 1:
    gap_start = (thetas[-1] + dtheta / 2.0) % (2 * np.pi)
    gap_end = (thetas[0] - dtheta / 2.0) % (2 * np.pi)
    if gap_end <= gap_start:
        gap_end += 2 * np.pi
else:
    gap_start = 0.0
    gap_end = 2 * np.pi

gap_span = gap_end - gap_start
gap_center = (gap_start + 0.5 * gap_span) % (2 * np.pi)

gap_span_deg = np.degrees(gap_span)
gap_center_deg = np.degrees(gap_center)
print(f"Gap center: {gap_center_deg:.1f}°, span: {gap_span_deg:.1f}°")


def subtract_gap_from_sector(a, b, g0, g1):
    """Return list of [start, end] segments from [a,b] with gap removed."""
    segs = [(a, b)]
    for gg0, gg1 in [(g0, g1), (g0 + 2*np.pi, g1 + 2*np.pi), (g0 - 2*np.pi, g1 - 2*np.pi)]:
        new_segs = []
        for s0, s1 in segs:
            if gg1 <= s0 or gg0 >= s1:
                new_segs.append((s0, s1))
            else:
                if gg0 > s0:
                    new_segs.append((s0, gg0))
                if gg1 < s1:
                    new_segs.append((gg1, s1))
        segs = new_segs
    return [(x0, x1) for x0, x1 in segs if x1 > x0]


# Radial layout outside existing family tiles
track_gap_1 = 0.06
heat_total = 0.5
band_h = heat_total / len(covariate_order)
r_heat_inner = r_outer + track_gap_1
r_heat_outer = r_heat_inner + heat_total

track_gap_2 = 0.05
bar_max = 0.22
r_bar_inner = r_heat_outer + track_gap_2
r_bar_outer = r_bar_inner + bar_max

# Heatmap ring from real PDP
cmap = sns.diverging_palette(220, 20, as_cmap=True)
norm = mcolors.TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)

for j, cov in enumerate(covariate_order):
    rr0 = r_heat_inner + j * band_h
    rr1 = rr0 + band_h * 0.96
    vals = pdp_matrix[cov].to_numpy()

    for i in range(n_species):
        a = theta_edges[i]
        b = theta_edges[i + 1]
        segs = subtract_gap_from_sector(a, b, gap_start, gap_end)
        for s0, s1 in segs:
            t1 = np.degrees(s0)
            t2 = np.degrees(s1)
            if t2 <= t1:
                t2 += 360
            w = Wedge(
                center=(0, 0),
                r=rr1,
                theta1=t1,
                theta2=t2,
                width=(rr1 - rr0),
                facecolor=cmap(norm(vals[i])),
                edgecolor="none",
                linewidth=0,
                alpha=0.95,
                zorder=6,
            )
            w.set_clip_on(True)
            ax_host.add_patch(w)

            if missing_pdp_mask[i]:
                tm = 0.5 * (s0 + s1)
                rm = 0.5 * (rr0 + rr1)
                ax_host.text(
                    rm * np.cos(tm),
                    rm * np.sin(tm),
                    "*",
                    fontsize=4.5,
                    ha="center",
                    va="center",
                    color="0.25",
                    zorder=7,
                )

# Stacked count bar ring from GBIF + ArHa points
# Heights are log-scaled on total counts; GBIF is lighter alpha, ArHa overlay is stronger.
gbif_vals = gbif_counts.to_numpy(dtype=float)
arha_vals = arha_counts.to_numpy(dtype=float)
total_vals = gbif_vals + arha_vals

gbif_log = np.log10(gbif_vals + 1.0)
total_log = np.log10(total_vals + 1.0)
count_scale_max = 6.0
gbif_log_clipped = np.clip(gbif_log, 0.0, count_scale_max)
total_log_clipped = np.clip(total_log, 0.0, count_scale_max)

gbif_alpha = max_alpha * 0.55
arha_alpha = min(1.0, max_alpha * 0.95)

for i in range(n_species):
    a = theta_edges[i]
    b = theta_edges[i + 1]
    segs = subtract_gap_from_sector(a, b, gap_start, gap_end)

    h_gbif = bar_max * (gbif_log_clipped[i] / count_scale_max)
    h_total = bar_max * (total_log_clipped[i] / count_scale_max)

    rr_gbif = r_bar_inner + h_gbif
    rr_total = r_bar_inner + h_total

    fam_i = species_to_family_norm.get(species_order_norm[i], "Unknown")
    bar_color = family_colors.get(fam_i, (0.55, 0.55, 0.55, 1.0))

    for s0, s1 in segs:
        t1 = np.degrees(s0)
        t2 = np.degrees(s1)
        if t2 <= t1:
            t2 += 360

        # Base GBIF segment (lighter)
        if h_gbif > 0:
            w_gbif = Wedge(
                center=(0, 0),
                r=rr_gbif,
                theta1=t1,
                theta2=t2,
                width=max(rr_gbif - r_bar_inner, 1e-6),
                facecolor=bar_color,
                edgecolor="k",
                linewidth=0.2,
                alpha=gbif_alpha,
                zorder=7,
            )
            w_gbif.set_clip_on(True)
            ax_host.add_patch(w_gbif)

        # ArHa overlay segment to reach total (stronger)
        if h_total > h_gbif:
            w_arha = Wedge(
                center=(0, 0),
                r=rr_total,
                theta1=t1,
                theta2=t2,
                width=max(rr_total - rr_gbif, 1e-6),
                facecolor=bar_color,
                edgecolor="k",
                linewidth=0.2,
                alpha=arha_alpha,
                zorder=8,
            )
            w_arha.set_clip_on(True)
            ax_host.add_patch(w_arha)

# baseline ring under stacked bars (keeps the visual radial guide)
for i in range(n_species):
    a = theta_edges[i]
    b = theta_edges[i + 1]
    segs = subtract_gap_from_sector(a, b, gap_start, gap_end)

    for s0, s1 in segs:
        t1 = np.degrees(s0)
        t2 = np.degrees(s1)
        if t2 <= t1:
            t2 += 360

        w_base = Wedge(
            center=(0, 0),
            r=r_bar_inner + 0.0015,
            theta1=t1,
            theta2=t2,
            width=0.0015,
            facecolor="none",
            edgecolor="grey",
            linewidth=0.2,
            alpha=0.9,
            zorder=9,
        )
        w_base.set_clip_on(True)
        ax_host.add_patch(w_base)

# Radial log axis for the stacked bar ring, anchored at the start of the opening
axis_angle = gap_start % (2 * np.pi)
axis_color = "0.30"
axis_tick_color = "0.35"
axis_label_color = "0.20"

ax_host.plot(
    [r_bar_inner * np.cos(axis_angle), r_bar_outer * np.cos(axis_angle)],
    [r_bar_inner * np.sin(axis_angle), r_bar_outer * np.sin(axis_angle)],
    color=axis_color,
    linewidth=0.8,
    zorder=10,
)

tick_exponents = list(range(0, 7))

# small angular tick width around the axis line
tick_half_width = 0.012
for tick_exp in tick_exponents:
    tick_height = bar_max * (tick_exp / count_scale_max)
    tick_r = r_bar_inner + tick_height
    tick_t1 = axis_angle - tick_half_width
    tick_t2 = axis_angle + tick_half_width

    tick = Wedge(
        center=(0, 0),
        r=tick_r + 0.0015,
        theta1=np.degrees(tick_t1),
        theta2=np.degrees(tick_t2),
        width=0.003,
        facecolor=axis_tick_color,
        edgecolor="none",
        alpha=0.95,
        zorder=10,
    )
    tick.set_clip_on(False)
    ax_host.add_patch(tick)

    tick_label_r = tick_r + 0.014
    ax_host.text(
        tick_label_r * np.cos(gap_center),
        tick_label_r * np.sin(gap_center),
        f"{tick_exp}",
        fontsize=6.8,
        ha="center",
        va="center",
        rotation=np.degrees(gap_center) - 90.0,
        rotation_mode="anchor",
        color=axis_label_color,
        zorder=11,
        clip_on=False,
    )
# Covariate labels in the gap (BIO equivalence + cleaned non-BIO labels)
covariate_label_map = {
    "EC2019_working": "Electricity Consumption",
    "F182013.v4c_web.avg_vis": "Night Lights",
    "wc2.1_30s_bio_1": "Annual Mean Temp",
    "wc2.1_30s_bio_2": "Mean Diurnal Range",
    "wc2.1_30s_bio_3": "Isothermality",
    "wc2.1_30s_bio_4": "Temp Seasonality",
    "wc2.1_30s_bio_5": "Max Temp Warmest Month",
    "wc2.1_30s_bio_6": "Min Temp Coldest Month",
    "wc2.1_30s_bio_7": "Temp Annual Range",
    "wc2.1_30s_bio_8": "Mean Temp Wettest Quarter",
    "wc2.1_30s_bio_9": "Mean Temp Driest Quarter",
    "wc2.1_30s_bio_10": "Mean Temp Warmest Quarter",
    "wc2.1_30s_bio_11": "Mean Temp Coldest Quarter",
    "wc2.1_30s_bio_12": "Annual Precip",
    "wc2.1_30s_bio_13": "Precip Wettest Month",
    "wc2.1_30s_bio_14": "Precip Driest Month",
    "wc2.1_30s_bio_15": "Precip Seasonality",
    "wc2.1_30s_bio_16": "Precip Wettest Quarter",
    "wc2.1_30s_bio_17": "Precip Driest Quarter",
    "wc2.1_30s_bio_18": "Precip Warmest Quarter",
    "wc2.1_30s_bio_19": "Precip Coldest Quarter",
    "gpw_v4_population_density_rev11_2020_30s": "Population Density",
    "wildareas-v3-2009-human-footprint_geo": "Human Footprint",
}
# Expand limits for outer tracks (keep equal aspect)
lim_now = max(
    abs(ax_host.get_xlim()[0]),
    abs(ax_host.get_xlim()[1]),
    abs(ax_host.get_ylim()[0]),
    abs(ax_host.get_ylim()[1]),
)
lim_new = max(lim_now, r_bar_outer * 1.10)
ax_host.set_xlim(-lim_new, lim_new)
ax_host.set_ylim(-lim_new, lim_new)
ax_host.set_aspect("equal", adjustable="box")

fig = ax_host.figure
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
label_font = FontProperties(family=plt.rcParams.get("font.family", None), size=7.0)


def draw_text_on_arc(ax, text, radius, theta_center, fontsize=7.0, color="0.20", zorder=8):
    text = str(text)
    if not text:
        return

    x0, y0 = ax.transData.transform((0.0, 0.0))
    xr, yr = ax.transData.transform((radius, 0.0))
    radius_px = np.hypot(xr - x0, yr - y0)
    if radius_px <= 0:
        return

    char_widths = np.array([
        renderer.get_text_width_height_descent(ch, label_font, ismath=False)[0]
        for ch in text
    ], dtype=float)
    total_angle = char_widths.sum() / radius_px
    max_angle = max(gap_span * 0.92, 1e-6)
    if total_angle > max_angle:
        char_widths *= max_angle / total_angle
        total_angle = max_angle

    theta = theta_center + 0.5 * total_angle
    for ch, char_w in zip(text, char_widths):
        dtheta = char_w / radius_px
        theta -= 0.5 * dtheta
        if not ch.isspace():
            ax.text(
                radius * np.cos(theta),
                radius * np.sin(theta),
                ch,
                fontsize=fontsize,
                ha="center",
                va="center",
                rotation=np.degrees(theta) - 90.0,
                rotation_mode="anchor",
                color=color,
                zorder=zorder,
                clip_on=False,
            )
        theta -= 0.5 * dtheta


for j, cov in enumerate(covariate_order):
    r_mid = r_heat_inner + (j + 0.5) * band_h
    draw_text_on_arc(
        ax_host,
        covariate_label_map.get(cov, cov),
        r_mid,
        gap_center,
        fontsize=7.0,
        color=axis_label_color,
        zorder=8,
    )

# Colorbar for Spearman ring
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
bbox = ax_host.get_position()
cax = fig.add_axes([min(bbox.x1 + 0.005, 0.82), bbox.y0 + 0.04, 0.12, 0.015])
cb = fig.colorbar(sm, cax=cax, orientation="horizontal")
cb.set_label("Spearman r", fontsize=9)
cb.ax.xaxis.set_label_position("top")
cb.ax.tick_params(labelsize=8)

plt.savefig("PDP_CircularPhylo.pdf", bbox_inches="tight", dpi=600)
plt.savefig("PDP_CircularPhylo.svg", bbox_inches="tight", dpi=600)
plt.show()
